
# COMM117 Workshop 10 — Advanced RAG, Chaining, Tool Use, and Routing

This workshop covers the last two lectures:
- RAG evaluation and diagnostics
- Advanced / modular RAG, variants, chaining, tool use, routing, and corrective verification

This **exercise notebook includes the toy documents and structured data inside the notebook**, so students do not need any extra files.


In [ ]:

# Basic imports
import math
import re
from collections import Counter

import pandas as pd


## 0. Toy knowledge base and structured data

In [ ]:

# Unstructured documents used throughout the workshop
DOCUMENTS = [
    {
        "doc_id": "policy_1",
        "title": "Refund Policy",
        "department": "support",
        "region": "global",
        "text": (
            "Refunds are available within 14 days of purchase for physical products in original condition. "
            "Digital products are non-refundable once downloaded. "
            "If a customer was charged twice, the duplicate charge can be refunded after verification. "
            "Subscription renewals are refundable only within 48 hours of renewal."
        ),
    },
    {
        "doc_id": "faq_1",
        "title": "Account FAQ",
        "department": "support",
        "region": "global",
        "text": (
            "Customers can reset their password using the account settings page. "
            "Two-factor authentication can be enabled in security settings. "
            "If login fails with error code 0xA93, the issue is usually related to authentication failure."
        ),
    },
    {
        "doc_id": "shipping_1",
        "title": "Shipping Policy",
        "department": "operations",
        "region": "global",
        "text": (
            "Standard shipping takes 3 to 5 business days. "
            "Express shipping takes 1 to 2 business days. "
            "Shipping fees are non-refundable after the order has been dispatched."
        ),
    },
    {
        "doc_id": "billing_1",
        "title": "Billing and Subscription",
        "department": "finance",
        "region": "global",
        "text": (
            "Monthly subscriptions renew automatically unless cancelled before the renewal date. "
            "Users can cancel from the billing settings page. "
            "Refund requests for subscription renewals must be submitted within 48 hours."
        ),
    },
    {
        "doc_id": "hr_uk_1",
        "title": "UK Leave Policy",
        "department": "hr",
        "region": "uk",
        "text": (
            "Full-time staff in the UK receive 25 days of annual leave per year, excluding public holidays. "
            "Unused leave may be carried over only with manager approval."
        ),
    },
    {
        "doc_id": "hr_us_1",
        "title": "US Leave Policy",
        "department": "hr",
        "region": "us",
        "text": (
            "Full-time staff in the US receive 15 days of paid time off per year. "
            "Public holidays are managed separately from PTO."
        ),
    },
    {
        "doc_id": "movie_1",
        "title": "The Dalton Girls",
        "department": "wiki",
        "region": "global",
        "text": "The Dalton Girls is a film directed by Reginald Le Borg.",
    },
    {
        "doc_id": "person_1",
        "title": "Reginald Le Borg",
        "department": "wiki",
        "region": "global",
        "text": "Reginald Le Borg died from a heart attack.",
    },
]

# Structured data used for tool-use questions
SALES_TABLE = {
    "Q1": {"actual": 100, "forecast": 95},
    "Q2": {"actual": 120, "forecast": 110},
    "Q3": {"actual": 105, "forecast": 108},
    "Q4": {"actual": 130, "forecast": 125},
}

pd.DataFrame(DOCUMENTS)[["doc_id", "title", "department", "region"]]


## 1. Chunk the documents

In [ ]:

def sentence_chunk(text, max_words=40):
    """Simple sentence-based chunking for the toy dataset."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    chunks = []
    current = []

    for sent in sentences:
        tentative = " ".join(current + [sent])
        if len(tentative.split()) <= max_words:
            current.append(sent)
        else:
            if current:
                chunks.append(" ".join(current))
            current = [sent]

    if current:
        chunks.append(" ".join(current))
    return chunks


In [ ]:
# TODO: build a chunk table called chunks_df.
# Each row should contain: chunk_id, doc_id, title, department, region, text
# Use sentence_chunk(doc["text"], max_words=40)

chunk_rows = []

# YOUR CODE HERE


chunks_df = pd.DataFrame(chunk_rows)
chunks_df

## 2. Baseline lexical retrieval

In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())


def simple_retrieve(query, chunks_df, top_k=3):
    """A tiny lexical retriever using token overlap."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: test the baseline retriever on a refund question.
results = simple_retrieve("Can I get a refund for a downloaded digital product?", chunks_df, top_k=3)

for r in results:
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']}")
    print(r["text"])
    print()


## 3. Retrieval evaluation

In [ ]:

EVAL_SET = [
    {
        "question": "Can I get a refund for a downloaded digital product?",
        "gold_chunk_ids": ["policy_1_chunk_0"],
    },
    {
        "question": "What does error code 0xA93 usually mean?",
        "gold_chunk_ids": ["faq_1_chunk_0"],
    },
    {
        "question": "How long does express shipping take?",
        "gold_chunk_ids": ["shipping_1_chunk_0"],
    },
    {
        "question": "How many days of annual leave do UK full-time staff receive?",
        "gold_chunk_ids": ["hr_uk_1_chunk_0"],
    },
]


In [ ]:
# TODO: implement retrieval metrics.
def hit_rate_at_k(results, gold_ids):
    # YOUR CODE HERE
    raise NotImplementedError


def precision_at_k(results, gold_ids, k):
    # YOUR CODE HERE
    raise NotImplementedError


def recall_at_k(results, gold_ids, k):
    # YOUR CODE HERE
    raise NotImplementedError


def reciprocal_rank(results, gold_ids):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

rows = []
for ex in EVAL_SET:
    results = simple_retrieve(ex["question"], chunks_df, top_k=3)
    rows.append({
        "question": ex["question"],
        "hit@3": hit_rate_at_k(results, ex["gold_chunk_ids"]),
        "precision@3": precision_at_k(results, ex["gold_chunk_ids"], 3),
        "recall@3": recall_at_k(results, ex["gold_chunk_ids"], 3),
        "MRR": reciprocal_rank(results, ex["gold_chunk_ids"]),
    })

pd.DataFrame(rows)


## 4. Query-transformed RAG

In [ ]:
def rewrite_query_simple(query):
    # TODO: rewrite a few common informal phrases to match document language.
    # Example ideas: "money back" -> "refund", "downloaded file" -> "digital product"
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: compare original and rewritten retrieval.
query = "Can I get my money back for a downloaded file?"
rewritten = rewrite_query_simple(query)

print("ORIGINAL:", query)
print("REWRITTEN:", rewritten)
print()

print("TOP RESULTS AFTER REWRITE")
for r in simple_retrieve(rewritten, chunks_df, top_k=3):
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']}")


## 5. Filtered RAG

In [ ]:
def filtered_retrieve(query, chunks_df, top_k=3, region=None, department=None):
    # TODO: filter the dataframe by metadata, then call simple_retrieve.
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: retrieve the UK leave policy using a region filter.
results = filtered_retrieve(
    "What is the annual leave policy?",
    chunks_df,
    top_k=3,
    region="uk",
    department="hr",
)

for r in results:
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']} | {r['region']}")
    print(r["text"])
    print()


## 6. Parent-child / hierarchical retrieval

In [ ]:
# In this toy notebook, the child chunk is the retrieved chunk.
# The parent document is the full source document.

def parent_document_lookup(doc_id):
    # TODO: return the full document dictionary matching doc_id from DOCUMENTS
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: retrieve a small chunk, then map it back to the full parent document.
child_results = simple_retrieve("digital products refund", chunks_df, top_k=1)
child = child_results[0]
parent_doc = parent_document_lookup(child["doc_id"])

print("CHILD CHUNK")
print(child["chunk_id"], "|", child["text"])
print()
print("PARENT DOCUMENT")
print(parent_doc["title"])
print(parent_doc["text"])


## 7. Multi-hop RAG

In [ ]:
def multi_hop_rag(question):
    # TODO:
    # 1) retrieve the director of The Dalton Girls
    # 2) set the intermediate entity to "Reginald Le Borg"
    # 3) retrieve how he died
    # 4) return a dict with question, hop1, intermediate_entity, hop2, answer
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: run the multi-hop workflow.
out = multi_hop_rag("Why did the director of The Dalton Girls die?")
print("QUESTION:", out["question"])
print("HOP 1:", out["hop1"]["text"])
print("INTERMEDIATE ENTITY:", out["intermediate_entity"])
print("HOP 2:", out["hop2"]["text"])
print("FINAL ANSWER:", out["answer"])


## 8. Tool use

In [ ]:
def calculator_tool(expression):
    # TODO: safely evaluate a simple arithmetic expression
    # YOUR CODE HERE
    raise NotImplementedError


def sales_delta_tool(quarter):
    # TODO: look up SALES_TABLE[quarter], compute actual - forecast, and return a short string
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: test the tools.
print("Calculator tool:", calculator_tool("90 - 20 - 35"))
print("Sales tool:", sales_delta_tool("Q2"))


## 9. Router

In [ ]:
def route_query(query):
    # TODO:
    # - sales / forecast / revenue / q1-q4 -> "sales_tool"
    # - arithmetic expressions -> "calculator"
    # - otherwise -> "rag"
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: inspect the router decisions.
queries = [
    "Can I get a refund for a downloaded digital product?",
    "Q2 actual versus forecast",
    "90 - 20 - 35",
]

for q in queries:
    print(q, "->", route_query(q))


## 10. Corrective / verification step

In [ ]:
def draft_answer_from_context(query, retrieved):
    # TODO: create a very simple rule-based draft answer from retrieved context
    # YOUR CODE HERE
    raise NotImplementedError


def verify_answer(answer, retrieved):
    # TODO: return True if the answer is supported by the retrieved context, else False
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: retrieve, draft, verify.
retrieved = simple_retrieve("Can I refund a downloaded digital product?", chunks_df, top_k=2)
draft = draft_answer_from_context("Can I refund a downloaded digital product?", retrieved)
supported = verify_answer(draft, retrieved)

print("DRAFT:", draft)
print("SUPPORTED:", supported)


## 11. End-to-end chained workflow

In [ ]:
def advanced_rag_workflow(query):
    # TODO:
    # 1) route the query
    # 2) if calculator -> use calculator_tool
    # 3) if sales_tool -> use sales_delta_tool
    # 4) else rewrite the query, retrieve, draft, verify, and return final answer
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:

# TODO: run the full workflow on three different query types.
for q in [
    "Can I get my money back for a downloaded file?",
    "Q2 actual versus forecast",
    "90 - 20 - 35",
]:
    print("=" * 80)
    print("QUERY:", q)
    out = advanced_rag_workflow(q)
    for k, v in out.items():
        print(k, ":", v)
